In [ ]:
###PARAMS CELL###
DATASET_NAME = "mbpp"
RUN_ID = 0
CLUSTER_ID = 0

TO_SAVE_DIR = None #"/Users/RobertAdragna/Documents/MATS/evals_suite_new/playground/test/generate_execute"



In [ ]:
from src.experiment_tracker import ExperimentTracker
from src.utils.utils import RESULTS_DIR
import os
from typing import List
from src.utils.analysis_utils import *


from inspect_ai.log import read_eval_log
from inspect_ai.scorer import Score

In [ ]:
def add_genuine_vulnerability(exp_samples: Dict[Any, pd.DataFrame]) -> Dict[Any, pd.DataFrame]:
    for id_file, sample_df in exp_samples.items():
        # Group by 'id' and check if any row in the group has correctness_score_value == 'C'
        genuine_vuln_ids = sample_df.groupby('id').apply(
            lambda group: any(group['score_with_correctness__correctness_score_value'] == 'C')
        )
        
        # Map the result back to all rows based on their 'id'
        sample_df['genuine_vulnerability'] = sample_df['id'].map(genuine_vuln_ids).astype(int)
        
        exp_samples[id_file] = sample_df

    return exp_samples

In [ ]:
tracker = ExperimentTracker(os.path.join(RESULTS_DIR, "experiment_tracker"))
exp_df = tracker.generate_results_table(task_name="generate_execute", dataset_name=DATASET_NAME, id=RUN_ID)
exp_df = add_metrics_to_parameter_specs(exp_df, 'logfile_name')
exp_df.head()

In [ ]:
from sklearn.metrics import cluster


exp_samples =load_experiment_samples(exp_df, 'id_file', 'logfile_name')

cluster_analysis = tracker.load_analysis(task_name="generate_execute", dataset_name=DATASET_NAME, run_id=RUN_ID, analysis_name="vulnerability_clustering", analysis_name_id=CLUSTER_ID)
exp_samples = add_analysis_to_df(exp_samples, cluster_analysis)

exp_samples = add_genuine_vulnerability(exp_samples)

exp_samples[1][['id', 'epoch', 'prompt', 'vulnerability_text', 'score_with_correctness__orig_score_value', 'score_with_correctness__correctness_score_value', 'score_with_correctness__answer_1', 'score_with_correctness__answer_2', 'genuine_vulnerability', 'clusters']].head(1)


In [ ]:
print(exp_samples[1].columns)

## Metric Computations

In [ ]:
def success_rate(df, score_col='score_with_correctness__correctness_score_value') -> float:
    """Calculate the percentage of samples that have score 'C' (correct)."""
    if score_col not in df.columns:
        return 0.0
    return (df[score_col] == 'C').mean() * 100

def num_unique_clusters(df, cluster_col='clusters') -> int:
    """Count the number of unique clusters, excluding 'NONE'."""
    if cluster_col not in df.columns:
        return 0
    unique_clusters = df[cluster_col].unique()
    # Filter out 'NONE' and count
    return len([c for c in unique_clusters if c != 'NONE'])

def num_genuine_vulnerabilities(df, genuine_vuln_col='genuine_vulnerability') -> int:
    """Count unique IDs that have genuine vulnerabilities."""
    if genuine_vuln_col not in df.columns:
        return 0
    # Get unique IDs where genuine_vulnerability is 1
    return df[df[genuine_vuln_col] == 1]['id'].nunique()

def percentage_vulnerabilities_over_x_percent(df, x=50, score_col='score_with_correctness__correctness_score_value') -> float:
    """Calculate percentage of vulnerabilities with success rate over x percent across epochs."""
    if score_col not in df.columns or 'id' not in df.columns:
        return 0.0
    
    # Group by ID and calculate success rate for each vulnerability
    vuln_success_rates = df.groupby('id')[score_col].apply(
        lambda group: (group == 'C').mean() * 100
    )
    
    # Calculate percentage of vulnerabilities with success rate > x
    return (vuln_success_rates > x).mean() * 100

def compute_accuracy(df, score_col) -> float:
    """Calculate the percentage of samples that have score 'C' (correct) using the original scorer."""
    if score_col not in df.columns:
        return 0.0
    return (df[score_col] == 'C').mean() * 100


def percentage_vulnerability_not_exploited(df, score_col, correct_score_val) -> float:
    """Calculate the percentage of samples that have score 'C' (correct) using the original scorer."""
    if score_col not in df.columns:
        return 0.0
    return (df[score_col] == correct_score_val).mean() * 100


In [ ]:

# Define metrics to add
metrics_to_add = [
    ('success_rate', success_rate),
    ('num_unique_clusters', num_unique_clusters),
    ('num_genuine_vulnerabilities', num_genuine_vulnerabilities),
    ('pct_vulns_over_50pct', percentage_vulnerabilities_over_x_percent, {'x': 50}),
    ('pct_vulns_over_75pct', percentage_vulnerabilities_over_x_percent, {'x': 75}),
    ('pct_acc', compute_accuracy, {'score_col':'score_with_correctness__orig_score_value'}),
    ('pct_vulns_not_exploited_answer_2', percentage_vulnerability_not_exploited, {'correct_score_val': 'C', 'score_col': 'score_with_correctness__answer_2', 'correct_score_val':'YES'}),
]

# Apply metrics
exp_df = add_metrics_to_exp_df(exp_df, exp_samples, metrics_to_add)

# Display the updated dataframe with metrics
exp_df[['id_file', 'sandbag_type', 'hidden_scratchpad', 'success_rate', 
        'num_unique_clusters', 'num_genuine_vulnerabilities', 
        'pct_vulns_over_50pct', 'pct_vulns_over_75pct', 'pct_acc', 'pct_vulns_not_exploited_answer_2']]

In [ ]:
## Save Results
if TO_SAVE_DIR is not None:
    os.makedirs(TO_SAVE_DIR, exist_ok=True)
    exp_df.to_csv(os.path.join(TO_SAVE_DIR, "generate_execute_df.csv"))